In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
### Shortcut for package import
from pkgimp import *

from nb2p import database, dataset, config
import nb2p.offline.dataset.kgtorrent as kgtorrent

tqdm.pandas()

In [3]:
DATASET_NAME = "distilkaggle"

In [ ]:
DIRS = config.dirs(DATASET_NAME)
DIRS.makedirs()

making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-ast
making dirs: /ssd/haotian/scs/distilkaggle
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-full
making dirs: /ssd/haotian/scs/distilkaggle/dfgtree
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-eda
making dirs: /ssd/haotian/scs/distilkaggle/logs/full
making dirs: /ssd/haotian/scs/distilkaggle/models/full
making dirs: /ssd/haotian/scs/distilkaggle/ipynb


In [5]:
# Connect to database

db, client = database.connect(dataset_name=DATASET_NAME, verbose=True)

Pinged to database nb2p-dk. You successfully connected to MongoDB!


# Preprocess code cells


In [6]:
RAW_DIR = DIRS.base / "raw"

In [ ]:
code_df = pd.read_csv(RAW_DIR / "code.csv")
code_df

In [ ]:
code_df = code_df.rename(columns={"source": "content", "cell_index": "cell_id"})
code_df[:3]

In [ ]:
from nb2p.parser import remove_comments_and_docstrings


def check_and_get_content_no_comment(content: str):
    try:
        return remove_comments_and_docstrings(str(content), "python")
    except Exception as e:
        # print(content)
        print(e)

    return pd.NA


code_df["content_no_comment"] = code_df["content"].progress_apply(
    lambda x: check_and_get_content_no_comment(x)
)
code_df[:3]

In [ ]:
for c in ["kernel_id", "cell_id"]:
    code_df = code_df[~pd.to_numeric(code_df[c], errors="coerce").isnull()]
    code_df[c] = pd.to_numeric(code_df[c], downcast="integer")

code_df

# (Optional) Save result to file to avoid losing processed data (as it's time-consuming)


In [7]:
# # ⚠️WARNING: This cell includes file writing

# code_df.to_csv(DIRS.base / "code_preprocessed.csv", index=False)

In [7]:
code_df = pd.read_csv(DIRS.base / "code_preprocessed.csv")
code_df

,kernel_id,cell_id,content,output_type,execution_count,content_no_comment
0,12034794,3,from mpl_toolkits.mplot3d import Axes3D\nfrom ...,NaN,1.0,from mpl_toolkits.mplot3d import Axes3D\nfrom ...
1,12034794,5,"for dirname, _, filenames in os.walk('/kaggle/...",stream,2.0,"for dirname, _, filenames in os.walk('/kaggle/..."
2,12034794,7,# Distribution graphs (histogram/bar graph) of...,NaN,3.0,"def plotPerColumnDistribution(df, nGraphShown,..."
3,12034794,8,# Correlation matrix\ndef plotCorrelationMatri...,NaN,4.0,"def plotCorrelationMatrix(df, graphWidth):\n ..."
4,12034794,9,# Scatter and density plots\ndef plotScatterMa...,NaN,5.0,"def plotScatterMatrix(df, plotSize, textSize):..."
...,...,...,...,...,...,...
12140204,203109,7,# From plot above we can make desicion how man...,stream,6.0,"plt.figure(figsize=(15, 10))\nplt.hist(df.msg...."
12140205,203109,8,# From plot above we can see that roughly spea...,stream,7.0,"plt.figure(figsize=(15, 10))\nplt.hist(df.msg...."
12140206,203109,9,"# Roughly speaking, the message contains 10 wo...",execute_result,8.0,from wordcloud import WordCloud\ncloud = WordC...
12140207,203109,10,"# At the end, let's show up how are messages d...",display_data,9.0,ds = df['2012-01-01': '2016-12-31']\nds.groupb...


# Check Segments

In [8]:
kernel_ids = list(code_df["kernel_id"].unique())

In [9]:
code_df[code_df["kernel_id"] == kernel_ids[20000]]

,kernel_id,cell_id,content,output_type,execution_count,content_no_comment
259599,13157217,2,# This Python 3 environment comes with many he...,stream,1.0,import numpy as np \nimport pandas as pd \nimp...
259600,13157217,3,from sklearn.neighbors import kneighbors_graph...,NaN,2.0,from sklearn.neighbors import kneighbors_graph...
259601,13157217,4,"import warnings\nwarnings.filterwarnings(""igno...",NaN,3.0,"import warnings\nwarnings.filterwarnings(""igno..."
259602,13157217,5,n_sample = 10**5\nlist_distribution_type = ['G...,stream,4.0,n_sample = 10**5\nlist_distribution_type = ['G...
259603,13157217,6,df_stat.to_csv('df_stat.csv')\ndf_stat_short.t...,NaN,5.0,df_stat.to_csv('df_stat.csv')\ndf_stat_short.t...
259604,13157217,7,"fig = plt.figure(figsize = (18, 8 ))\nfig.add_...",display_data,6.0,"fig = plt.figure(figsize = (18, 8 ))\nfig.add_..."
259605,13157217,8,fig.savefig('hubs_positions'),NaN,7.0,fig.savefig('hubs_positions')
259606,13157217,9,NaN,NaN,NaN,NaN


In [10]:
kcid_df = code_df[["kernel_id", "cell_id"]]
kcid_df

,kernel_id,cell_id
0,12034794,3
1,12034794,5
2,12034794,7
3,12034794,8
4,12034794,9
...,...,...
12140204,203109,7
12140205,203109,8
12140206,203109,9
12140207,203109,10


In [11]:
def is_prompted(cell_ids: List[int]):
    if len(cell_ids) == 1:
        return False

    result = False
    for i in range(len(cell_ids) - 1):
        if cell_ids[i + 1] - cell_ids[i] > 1:
            result = True

    return result

In [12]:
kernel_id_dfs = {}
for k, v in kcid_df.groupby(["kernel_id"]):
    kernel_id_dfs[k[0]] = v

In [13]:
code_dfs = {}
for k, v in code_df.groupby(["kernel_id"]):
    code_dfs[k[0]] = v

In [14]:
from typing import Tuple
from tqdm.contrib.concurrent import process_map
import tqdm


def para_prompted(d: Tuple[int, pd.DataFrame]):
    kernel_id, df = d
    cell_ids: List[int] = list(df["cell_id"])

    return {"notebook_id": kernel_id, "prompted": is_prompted(cell_ids)}


prompted_result = process_map(
    para_prompted,
    kernel_id_dfs.items(),
    tqdm_class=tqdm.notebook.tqdm,
    chunksize=1000,
    max_workers=os.cpu_count(),
)

  0%|          | 0/510534 [00:00<?, ?it/s]

In [15]:
def para_get_cells_dict(d: Tuple[int, pd.DataFrame]):
    kernel_id, df = d
    df = df.drop(columns=["kernel_id"])

    return {"notebook_id": kernel_id, "cells": df}


cells_dict = process_map(
    para_get_cells_dict,
    code_dfs.items(),
    tqdm_class=tqdm.notebook.tqdm,
    chunksize=1000,
    max_workers=os.cpu_count(),
)

  0%|          | 0/510534 [00:00<?, ?it/s]

In [16]:
prompted_result[:10]

[{'notebook_id': 17375, 'prompted': False},
 {'notebook_id': 17416, 'prompted': False},
 {'notebook_id': 17477, 'prompted': False},
 {'notebook_id': 17485, 'prompted': False},
 {'notebook_id': 17487, 'prompted': False},
 {'notebook_id': 17537, 'prompted': False},
 {'notebook_id': 17548, 'prompted': False},
 {'notebook_id': 17648, 'prompted': False},
 {'notebook_id': 17884, 'prompted': False},
 {'notebook_id': 17896, 'prompted': False}]

In [17]:
def get_segment_ends(cell_ids: List[int]):
    result = []

    for i in range(len(cell_ids) - 1):
        if cell_ids[i + 1] - cell_ids[i] > 1:
            result.append(i)

    return result


# NOTE: the final index will NEVER appear in the result
# This is why we can safely + 1 to get the end range vars
get_segment_ends([1, 2, 4, 5, 8])

[1, 3]

In [18]:
from typing import TypeVar
from itertools import chain

T = TypeVar("T")


def get_segments(cells: List[T], ends: List[int]) -> List[List[T]]:
    end_range_vars = [e + 1 for e in ends]

    pairs = zip(chain([0], end_range_vars), chain(end_range_vars, [None]))
    return [cells[i:j] for i, j in pairs]


get_segments(["1st", "2nd", "3rd", "4th", "5th"], get_segment_ends([1, 2, 4, 5, 8]))

[['1st', '2nd'], ['3rd', '4th'], ['5th']]

In [19]:
from nb2p import fileop

In [20]:
cells_dict[997]

{'notebook_id': 44435,
 'cells':           cell_id                                            content  \
 11610624        2  import numpy as np\nimport matplotlib.pyplot a...   
 11610625        4  def plot_history(history):\n    loss_list = [s...   
 11610626        6  def plot_confusion_matrix(cm, classes,\n      ...   
 11610627        8  iris = datasets.load_iris()\nx = iris.data\ny ...   
 11610628       10  x_train, x_test, y_train, y_test = train_test_...   
 11610629       12  model = Sequential()\nmodel.add(Dense(8,activa...   
 11610630       13                              plot_history(history)   
 11610631       15  full_multiclass_report(model,\n               ...   
 11610632       17  full_multiclass_report(model,\n               ...   
 11610633       19  y = iris.target\n\nx_train, x_test, y_train, y...   
 11610634       21  ## First redefine y as categorical variable\ny...   
 11610635       23                                                NaN   
 
          output_

# Save to DB


In [21]:
db.notebook.drop()
db.notebook.create_index(["notebook_id"])

'notebook_id_1'

In [22]:
db.segment.drop()
db.segment.create_index(["notebook_id"])

'notebook_id_1'

In [23]:
db.cell.drop()
db.cell.create_index(["notebook_id"])

'notebook_id_1'

In [24]:
db.notebook.insert_many(prompted_result) is None

False

In [25]:
import pandas as pd
from pymongo.database import Database as PyMongoDB


def get_notebook_object_id(db: PyMongoDB, notebook_id: int):
    result = db.notebook.find_one({"notebook_id": notebook_id})
    if not result:
        raise ValueError(f"cannot find notebook_id: {notebook_id}")

    return result["_id"]


def is_invalid_cell_value(v):
    return (not v) or isinstance(v, float) or (v == np.nan)


def is_valid_list(v):
    return v and isinstance(v, list)


def split_code_lines(v: str):
    if is_invalid_cell_value(v):
        return []
    else:
        return v.split("\n")


def make_cell_record_dicts(d):
    db, client = database.connect(dataset_name=DATASET_NAME)

    nb_id, v = d.values()
    cells_dict = v.to_dict(orient="records")

    nb_obj_id = get_notebook_object_id(db, nb_id)

    test_processed = db.cell.find_one({"notebook_id": nb_obj_id})
    if test_processed:
        return

    with client.start_session() as session:
        with session.start_transaction():
            is_prompted = db.notebook.find_one({"_id": nb_obj_id})["prompted"]

            cells_dict = [
                c for c in cells_dict if not is_invalid_cell_value(c["content"])
            ]

            if not cells_dict:
                print(f"WARN  notebook {nb_id} ({nb_obj_id}) is invalid")
                db.notebook.update_one({"_id": nb_obj_id}, {"$set": {"cells": []}})
                return

            for cell in cells_dict:
                cell["content"] = split_code_lines(cell["content"])
                cell["content_no_comment"] = split_code_lines(
                    cell["content_no_comment"]
                )
                cell["notebook_id"] = nb_obj_id

            result = db.cell.insert_many(cells_dict)
            cell_obj_ids = result.inserted_ids
            db.notebook.update_one(
                {"_id": nb_obj_id}, {"$set": {"cells": cell_obj_ids}}
            )

            if not is_prompted:
                return

            cell_ids = [c["cell_id"] for c in cells_dict]
            segments = get_segments(cell_obj_ids, get_segment_ends(cell_ids))
            # print(len(cells_dict), len(segments), segments)

            segment_obj_ids = []
            for i, s in enumerate(segments):
                result = db.segment.insert_one(
                    {
                        "cells": s,
                        "has_code": True,
                        "notebook_id": nb_obj_id,
                        "segment_id": i,
                    }
                )
                segment_obj_ids.append(result.inserted_id)

            db.notebook.update_one(
                {"_id": nb_obj_id}, {"$set": {"segments": segment_obj_ids}}
            )

In [26]:
# make_cell_record_dicts(cells_dict[997])

In [27]:
process_map(
    make_cell_record_dicts,
    cells_dict,
    chunksize=1000,
    max_workers=os.cpu_count(),
) is None

WARN  notebook 17537 (66f3ce8e975730f2dded4f1a) is invalid


  0%|          | 0/510534 [00:00<?, ?it/s]

WARN  notebook 93491 (66f3ce8e975730f2dded5ec3) is invalid
WARN  notebook 93546 (66f3ce8e975730f2dded5ec8) is invalid
WARN  notebook 144327 (66f3ce8e975730f2dded6e55) is invalid
WARN  notebook 93807 (66f3ce8e975730f2dded5ed6) is invalid
WARN  notebook 21609 (66f3ce8e975730f2dded4f57) is invalid
WARN  notebook 64817 (66f3ce8e975730f2dded5714) is invalid
WARN  notebook 200943 (66f3ce8e975730f2dded7df9) is invalid
WARN  notebook 22996 (66f3ce8e975730f2dded4f77) is invalid
WARN  notebook 145252 (66f3ce8e975730f2dded6e74) is invalid
WARN  notebook 125355 (66f3ce8e975730f2dded6aa1) is invalid
WARN  notebook 23276 (66f3ce8e975730f2dded4f83) is invalid
WARN  notebook 82448 (66f3ce8e975730f2dded5b21) is invalid
WARN  notebook 273451 (66f3ce8e975730f2dded8d98) is invalid
WARN  notebook 258608 (66f3ce8e975730f2dded89bb) is invalid
WARN  notebook 163543 (66f3ce8e975730f2dded727a) is invalid
WARN  notebook 221424 (66f3ce8e975730f2dded81f7) is invalid
WARN  notebook 146194 (66f3ce8e975730f2dded6e9d)

False

In [28]:
for nb in db.notebook.find():
    if len(nb["cells"]) == 0:
        print(nb["_id"])

        db.notebook.update_one({"_id": nb["_id"]}, {"$set": {"can_parse": False}})
    else:
        db.notebook.update_one({"_id": nb["_id"]}, {"$set": {"can_parse": True}})

66f3ce8e975730f2dded4f1a
66f3ce8e975730f2dded4f57
66f3ce8e975730f2dded4f77
66f3ce8e975730f2dded4f83
66f3ce8e975730f2dded4fb1
66f3ce8e975730f2dded4fb4
66f3ce8e975730f2dded4fb7
66f3ce8e975730f2dded4fd2
66f3ce8e975730f2dded4feb
66f3ce8e975730f2dded4ff5
66f3ce8e975730f2dded5024
66f3ce8e975730f2dded5071
66f3ce8e975730f2dded5089
66f3ce8e975730f2dded508b
66f3ce8e975730f2dded5096
66f3ce8e975730f2dded50f5
66f3ce8e975730f2dded51b5
66f3ce8e975730f2dded51c9
66f3ce8e975730f2dded522c
66f3ce8e975730f2dded52cb
66f3ce8e975730f2dded54c0
66f3ce8e975730f2dded5524
66f3ce8e975730f2dded5539
66f3ce8e975730f2dded5560
66f3ce8e975730f2dded5568
66f3ce8e975730f2dded5587
66f3ce8e975730f2dded55d7
66f3ce8e975730f2dded564b
66f3ce8e975730f2dded5714
66f3ce8e975730f2dded5799
66f3ce8e975730f2dded5809
66f3ce8e975730f2dded583b
66f3ce8e975730f2dded5866
66f3ce8e975730f2dded589a
66f3ce8e975730f2dded58c1
66f3ce8e975730f2dded58f5
66f3ce8e975730f2dded593b
66f3ce8e975730f2dded5999
66f3ce8e975730f2dded59cc
66f3ce8e975730f2dded59e5


In [29]:
db.cell.update_many({}, {"$set": {"type": "code"}})

In [6]:
from nb2p.dataset import materialize


materialize(db)